# Disparity Analysis Audit Program

This program assesses disparate impact under equalized odds framework by comparing
coverage and annotation quality metrics across target groups within the same coding level.

In [ ]:
# Imports
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import pandas as pd
import re

In [ ]:
@dataclass
class DisparityAuditConfig:
    # `workdir` is the anchor used to resolve all relative paths.
    workdir: Path

    # Local sources expected to exist in the repository.
    data_local_path: Path = Path('../outputs/preprocessing/dedup_primary.tsv')
    glossary_local_path: Path = Path('../data/glossary.tsv')

    # Directory where audit exports are written.
    output_dir: Path = Path('../outputs/disparity_audits')

In [ ]:
# Configuration
WORKDIR = Path.cwd()
cfg = DisparityAuditConfig(workdir=WORKDIR)

# Convert all configured relative paths into absolute paths once up front.
cfg.output_dir = cfg.workdir / cfg.output_dir
cfg.data_path = cfg.workdir / cfg.data_local_path
cfg.glossary_path = cfg.workdir / cfg.glossary_local_path

# Create output/cache directories early so later cells can assume they exist.
cfg.output_dir.mkdir(parents=True, exist_ok=True)

cfg

In [ ]:
# Load data
glossary = pd.read_csv(cfg.glossary_path, sep='\t')
data = pd.read_csv(cfg.data_path, sep='\t')

# Ensure everything is lowercase for matching
glossary['surface_form'] = glossary['surface_form'].str.strip().str.lower()
glossary['target'] = glossary['target'].str.strip().str.lower()

# Compile regex: Sorting by length (descending) prevents partial matches 
# (e.g., matching 'dog' inside 'dogwhistle')
all_forms = sorted(glossary['surface_form'].unique(), key=len, reverse=True)
pattern = re.compile(r'\b(' + '|'.join(map(re.escape, all_forms)) + r')\b', flags=re.IGNORECASE)

In [ ]:
# 1. Extract matches
data['found_forms'] = data['text'].apply(lambda x: pattern.findall(x.lower()) if pd.notna(x) else [])

# 2. Explode and Join
# Each row in 'matches_df' represents one instance of a found dogwhistle
matches_df = data.explode('found_forms').dropna(subset=['found_forms'])

# 3. Merge with glossary to get the taxonomy_level and target for each match
# This handles cases where one surface_form might belong to multiple categories
audit_df = matches_df.merge(
    glossary, 
    left_on='found_forms', 
    right_on='surface_form', 
    how='inner'
)

In [ ]:
def compute_coverage_metrics(audit_df: pd.DataFrame, glossary: pd.DataFrame) -> pd.DataFrame:
    """Compute coverage metrics (presence rates, type coverage, etc.)"""
    metrics_data = []

    for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
        # Get glossary reference: all forms and types for this level/target combination
        glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                    (glossary['target'] == target)]
        total_glossary_forms = glossary_subset['surface_form'].nunique()
        total_glossary_types = glossary_subset['type'].nunique()

        # Calculate metrics
        # Presence rate: proportion of distinct surface forms found vs. glossary
        distinct_forms_found = group['found_forms'].nunique()
        presence_rate = distinct_forms_found / total_glossary_forms if total_glossary_forms > 0 else 0

        # Type coverage: proportion of distinct dogwhistle categories (types) represented
        distinct_types_found = group['type'].nunique()
        type_coverage = distinct_types_found / total_glossary_types if total_glossary_types > 0 else 0

        # Token frequency: total count of matched instances
        total_tokens = len(group)

        metrics_data.append({
            'taxonomy_level': level,
            'target': target,
            'total_glossary_forms': total_glossary_forms,
            'total_glossary_types': total_glossary_types,
            'distinct_forms_found': distinct_forms_found,
            'distinct_types_found': distinct_types_found,
            'presence_rate': presence_rate,
            'type_coverage': type_coverage,
            'token_frequency': total_tokens
        })

    return pd.DataFrame(metrics_data)

In [ ]:
def compute_annotation_quality_metrics(audit_df: pd.DataFrame, data: pd.DataFrame, glossary: pd.DataFrame) -> pd.DataFrame:
    """Compute annotation quality metrics (cases A/B/C, labeling rates)"""
    annotation_quality_data = []

    for (level, target), group in audit_df.groupby(['taxonomy_level', 'target']):
        # Case A: Posts with forms from this level/target labeled as hateful
        case_a = len(group[group['binary_hate'] == 1])

        # Case B: Posts with forms from this level/target labeled as non-hateful
        case_b = len(group[group['binary_hate'] == 0])

        # Case C: Posts that target this group but DON'T contain any forms 
        # from this level/target combination
        target_group_posts = data[data['targets'].str.contains(target, case=False, na=False)]

        # Find posts that contain any form from this glossary subset
        glossary_subset = glossary[(glossary['taxonomy_level'] == level) & 
                                    (glossary['target'] == target)]
        forms_for_level_target = set(glossary_subset['surface_form'].unique())

        # Posts in target group that have forms from this level/target
        posts_with_forms = group['text_dedup_key'].unique()

        # Case C: Target group posts without forms from this level/target
        case_c = len(target_group_posts[~target_group_posts['text_dedup_key'].isin(posts_with_forms)])

        # Calculate metrics
        matches_with_label = case_a + case_b
        correct_labeling_rate = case_a / matches_with_label if matches_with_label > 0 else 0
        annotator_failure_ratio = case_b / matches_with_label if matches_with_label > 0 else 0

        annotation_quality_data.append({
            'taxonomy_level': level,
            'target': target,
            'case_a_present_hateful': case_a,
            'case_b_present_nonhateful': case_b,
            'case_c_absent': case_c,
            'correct_labeling_rate': correct_labeling_rate,
            'annotator_failure_ratio': annotator_failure_ratio,
            'total_matches': matches_with_label,
            'total_target_group_posts': len(target_group_posts)
        })

    return pd.DataFrame(annotation_quality_data)

In [ ]:
# Compute coverage and annotation quality metrics
coverage_report = compute_coverage_metrics(audit_df, glossary)
annotation_quality_report = compute_annotation_quality_metrics(audit_df, data, glossary)

print("Coverage Report:")
print(coverage_report)
print("\nAnnotation Quality Report:")
print(annotation_quality_report)

In [ ]:
# Coverage Disparity Analysis: Compare metrics across target groups within same level
coverage_disparities = []

for level in coverage_report['taxonomy_level'].unique():
    level_data = coverage_report[coverage_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Presence rate disparity
                presence_gap = data_a['presence_rate'] - data_b['presence_rate']
                presence_gap_abs = abs(presence_gap)

                # Type coverage disparity
                type_gap = data_a['type_coverage'] - data_b['type_coverage']
                type_gap_abs = abs(type_gap)

                # Token frequency disparity (relative)
                if data_b['token_frequency'] > 0:
                    token_ratio = data_a['token_frequency'] / data_b['token_frequency']
                else:
                    token_ratio = float('inf') if data_a['token_frequency'] > 0 else 1.0

                coverage_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'presence_rate_a': data_a['presence_rate'],
                    'presence_rate_b': data_b['presence_rate'],
                    'presence_rate_gap': presence_gap,
                    'presence_rate_gap_abs': presence_gap_abs,
                    'type_coverage_a': data_a['type_coverage'],
                    'type_coverage_b': data_b['type_coverage'],
                    'type_coverage_gap': type_gap,
                    'type_coverage_gap_abs': type_gap_abs,
                    'token_freq_a': data_a['token_frequency'],
                    'token_freq_b': data_b['token_frequency'],
                    'token_freq_ratio': token_ratio
                })

coverage_disparity_report = pd.DataFrame(coverage_disparities)

In [ ]:
# Annotation Quality Disparity Analysis: Compare labeling metrics across target groups within same level
annotation_disparities = []

for level in annotation_quality_report['taxonomy_level'].unique():
    level_data = annotation_quality_report[annotation_quality_report['taxonomy_level'] == level]
    targets = level_data['target'].tolist()

    if len(targets) > 1:
        # Compare each pair of targets within this level
        for i in range(len(targets)):
            for j in range(i+1, len(targets)):
                target_a = targets[i]
                target_b = targets[j]

                data_a = level_data[level_data['target'] == target_a].iloc[0]
                data_b = level_data[level_data['target'] == target_b].iloc[0]

                # Correct labeling rate disparity
                labeling_gap = data_a['correct_labeling_rate'] - data_b['correct_labeling_rate']
                labeling_gap_abs = abs(labeling_gap)

                # Annotator failure ratio disparity
                failure_gap = data_a['annotator_failure_ratio'] - data_b['annotator_failure_ratio']
                failure_gap_abs = abs(failure_gap)

                annotation_disparities.append({
                    'taxonomy_level': level,
                    'target_a': target_a,
                    'target_b': target_b,
                    'correct_labeling_rate_a': data_a['correct_labeling_rate'],
                    'correct_labeling_rate_b': data_b['correct_labeling_rate'],
                    'labeling_rate_gap': labeling_gap,
                    'labeling_rate_gap_abs': labeling_gap_abs,
                    'annotator_failure_ratio_a': data_a['annotator_failure_ratio'],
                    'annotator_failure_ratio_b': data_b['annotator_failure_ratio'],
                    'failure_ratio_gap': failure_gap,
                    'failure_ratio_gap_abs': failure_gap_abs,
                    'case_a_a': data_a['case_a_present_hateful'],
                    'case_a_b': data_b['case_a_present_hateful'],
                    'case_b_a': data_a['case_b_present_nonhateful'],
                    'case_b_b': data_b['case_b_present_nonhateful']
                })

annotation_disparity_report = pd.DataFrame(annotation_disparities)

In [ ]:
# Print results
print("=" * 120)
print("DISPARITY ANALYSIS AUDIT")
print("=" * 120)

if len(coverage_disparity_report) > 0:
    print("\n" + "=" * 120)
    print("COVERAGE DISPARITY ANALYSIS: Within-Level Comparisons")
    print("=" * 120)
    print("\nPresence Rate Disparities (higher = target_a has better coverage):")
    presence_cols = ['taxonomy_level', 'target_a', 'target_b', 'presence_rate_a', 'presence_rate_b', 'presence_rate_gap', 'presence_rate_gap_abs']
    print(coverage_disparity_report[presence_cols].to_string(index=False))

    print("\nType Coverage Disparities (higher = target_a has better type representation):")
    type_cols = ['taxonomy_level', 'target_a', 'target_b', 'type_coverage_a', 'type_coverage_b', 'type_coverage_gap', 'type_coverage_gap_abs']
    print(coverage_disparity_report[type_cols].to_string(index=False))

    print("\nToken Frequency Ratios (ratio > 1 = target_a appears more frequently):")
    token_cols = ['taxonomy_level', 'target_a', 'target_b', 'token_freq_a', 'token_freq_b', 'token_freq_ratio']
    print(coverage_disparity_report[token_cols].to_string(index=False))

if len(annotation_disparity_report) > 0:
    print("\n" + "=" * 120)
    print("ANNOTATION QUALITY DISPARITY ANALYSIS: Within-Level Comparisons")
    print("=" * 120)
    print("\nCorrect Labeling Rate Disparities (higher = target_a has better annotation quality):")
    labeling_cols = ['taxonomy_level', 'target_a', 'target_b', 'correct_labeling_rate_a', 'correct_labeling_rate_b', 'labeling_rate_gap', 'labeling_rate_gap_abs']
    print(annotation_disparity_report[labeling_cols].to_string(index=False))

    print("\nAnnotator Failure Ratio Disparities (higher = target_a has worse annotation quality):")
    failure_cols = ['taxonomy_level', 'target_a', 'target_b', 'annotator_failure_ratio_a', 'annotator_failure_ratio_b', 'failure_ratio_gap', 'failure_ratio_gap_abs']
    print(annotation_disparity_report[failure_cols].to_string(index=False))

    print("\nCase Counts for Context:")
    case_cols = ['taxonomy_level', 'target_a', 'target_b', 'case_a_a', 'case_b_a', 'case_a_b', 'case_b_b']
    print(annotation_disparity_report[case_cols].to_string(index=False))

In [ ]:
# Export disparity analysis results
if len(coverage_disparity_report) > 0:
    coverage_disparity_path = cfg.output_dir / 'coverage_disparity.tsv'
    coverage_disparity_report.to_csv(coverage_disparity_path, sep='\t', index=False)
    print(f"\nCoverage disparity analysis exported to: {coverage_disparity_path}")

if len(annotation_disparity_report) > 0:
    annotation_disparity_path = cfg.output_dir / 'annotation_disparity.tsv'
    annotation_disparity_report.to_csv(annotation_disparity_path, sep='\t', index=False)
    print(f"Annotation disparity analysis exported to: {annotation_disparity_path}")

print(f"\nAll disparity analysis results exported to: {cfg.output_dir}")